# 25. Unsupervised Learning: Hierarchical Clustering

## Algorithm Category
**Type**: Unsupervised Learning - Clustering  
**Complexity**: Medium  
**Use Case**: Build hierarchy of clusters using agglomerative or divisive approach

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand hierarchical clustering and its variants (agglomerative, divisive)
- Implement agglomerative hierarchical clustering
- Understand linkage criteria (single, complete, average, ward)
- Visualize dendrograms
- Extract clusters at different levels
- Apply hierarchical clustering to real-world problems

## Historical Context

Hierarchical clustering has been used since the 1950s:
- Ward, J.H. (1963): "Hierarchical grouping to optimize an objective function"
- Johnson, S.C. (1967): "Hierarchical clustering schemes"
- One of the oldest clustering methods

**Key Papers/References:**
- Ward, J.H. (1963). "Hierarchical grouping to optimize an objective function"
- Johnson, S.C. (1967). "Hierarchical clustering schemes"

## When to Use Hierarchical Clustering

Hierarchical clustering is appropriate when:
- You want to explore cluster structure at multiple levels
- You don't know the number of clusters in advance
- You need to visualize cluster relationships
- Clusters may have nested structure
- You want interpretable cluster hierarchy
- Working with small to medium datasets

## Theory & Mechanics

### Mathematical Foundation

Hierarchical clustering builds a tree of clusters (dendrogram).

**Agglomerative (Bottom-Up):**
1. Start with each point as its own cluster
2. Merge closest clusters iteratively
3. Continue until all points in one cluster

**Divisive (Top-Down):**
1. Start with all points in one cluster
2. Split clusters iteratively
3. Continue until each point is its own cluster

**Linkage Criteria:**

1. **Single Linkage (Minimum):**
   $$d(C_i, C_j) = \min_{x \in C_i, y \in C_j} ||x - y||$$

2. **Complete Linkage (Maximum):**
   $$d(C_i, C_j) = \max_{x \in C_i, y \in C_j} ||x - y||$$

3. **Average Linkage:**
   $$d(C_i, C_j) = \frac{1}{|C_i||C_j|} \sum_{x \in C_i} \sum_{y \in C_j} ||x - y||$$

4. **Ward Linkage (Minimizes variance):**
   $$d(C_i, C_j) = \frac{|C_i||C_j|}{|C_i| + |C_j|} ||\mu_i - \mu_j||^2$$

### How It Works

**Agglomerative Algorithm:**
1. **Initialize**: Each point is a cluster
2. **Compute distances**: Calculate distance matrix between clusters
3. **Merge**: Merge two closest clusters
4. **Update**: Update distance matrix
5. **Repeat**: Steps 2-4 until one cluster remains

### Key Hyperparameters

- **n_clusters**: Number of clusters to extract (if specified)
- **linkage**: Linkage criterion ('ward', 'complete', 'average', 'single')
- **affinity**: Distance metric ('euclidean', 'manhattan', 'cosine', etc.)
- **distance_threshold**: Cut-off distance for flat clustering

### Advantages

- No need to specify number of clusters
- Produces interpretable dendrogram
- Works with any distance metric
- Can extract clusters at any level
- Handles non-spherical clusters better than K-Means

### Limitations

- Computationally expensive O(n³) for agglomerative
- Sensitive to noise and outliers
- Greedy algorithm (may not find global optimum)
- Difficult to scale to large datasets
- Memory intensive (stores full distance matrix)


## Implementation

Let's implement hierarchical clustering and visualize dendrograms.


In [ ]:
# ============================================
# IMPORTING LIBRARIES: Setting Up Our Tools
# ============================================

# Core data science libraries
import numpy as np  # NumPy: Numerical computing (arrays, math operations)
import pandas as pd  # Pandas: Data manipulation (DataFrames, data analysis)
import matplotlib.pyplot as plt  # Matplotlib: Plotting and visualization

# Scikit-learn: Machine learning library
from sklearn.datasets import (
    make_blobs,  # Generate synthetic blob-shaped clusters (for demonstration)
    load_iris  # Iris flower dataset (for real-world example)
)
from sklearn.cluster import AgglomerativeClustering  # Agglomerative hierarchical clustering
from sklearn.preprocessing import StandardScaler  # Feature scaling
from scipy.cluster.hierarchy import (
    dendrogram,  # Visualize hierarchical clustering as tree (dendrogram)
    linkage  # Compute linkage matrix for hierarchical clustering
)
from sklearn.metrics import silhouette_score  # Calculate silhouette score (cluster quality metric)

# ============================================
# IMPORTING OUR HELPER FUNCTIONS
# ============================================

# Our custom utility functions (organized in src/ directory)
from src.models.unsupervised import (
    hierarchical_cluster,  # Hierarchical clustering wrapper function
    evaluate_clustering  # Evaluate clustering quality (silhouette score)
)

print("Libraries imported successfully!")  # Confirm all imports worked


In [ ]:
# ============================================
# GENERATING SYNTHETIC DATASET: For Demonstration
# ============================================

# make_blobs() generates synthetic data with known cluster structure
# This is useful for understanding how hierarchical clustering works
# In real applications, you'd load your own data

# make_blobs() parameters:
# n_samples=150: Number of data points to generate
# centers=4: Number of clusters (blobs) to create
# n_features=2: Number of features (2D data for easy visualization)
# random_state=42: Ensures reproducible results
# cluster_std=0.60: Standard deviation of clusters (controls how spread out they are)
X, y_true = make_blobs(n_samples=150, centers=4, n_features=2, 
                       random_state=42, cluster_std=0.60)
# Returns:
# - X: Feature values (150 samples × 2 features)
# - y_true: True cluster labels (for comparison - we won't use these for clustering!)

print(f"Dataset Shape: {X.shape}")  # Output: (150, 2) - 150 points, 2 features
print(f"True number of clusters: {len(np.unique(y_true))}")  # Output: 4 clusters

# ============================================
# APPLYING AGGLOMERATIVE CLUSTERING
# ============================================

# AgglomerativeClustering builds clusters bottom-up (each point starts as its own cluster)
# n_clusters=4: Extract 4 clusters from the hierarchy
# linkage='ward': Linkage criterion (minimizes variance when merging clusters)
#   - 'ward': Minimizes variance (good for spherical clusters)
#   - 'complete': Maximum distance between clusters
#   - 'average': Average distance between clusters
#   - 'single': Minimum distance between clusters
clustering = AgglomerativeClustering(n_clusters=4, linkage='ward')

# fit_predict() trains the model and returns cluster assignments
y_pred = clustering.fit_predict(X)
# Returns: array of cluster labels (0, 1, 2, or 3 for each sample)

print(f"\nHierarchical Clustering Results:")
print(f"  Number of clusters: {clustering.n_clusters}")  # Number of clusters (4)
print(f"  Linkage: {clustering.linkage}")  # Linkage method used ('ward')

# ============================================
# VISUALIZING CLUSTERING RESULTS
# ============================================

# Create figure with 2 subplots
plt.figure(figsize=(12, 5))  # Width=12 inches, height=5 inches

# Subplot 1: True clusters (ground truth - what we're trying to discover)
plt.subplot(1, 2, 1)  # 1 row, 2 columns, position 1 (left)

# Scatter plot: X[:, 0] = first feature, X[:, 1] = second feature
# c=y_true colors points by true cluster labels
plt.scatter(X[:, 0], X[:, 1], c=y_true, cmap='viridis', s=50, alpha=0.7)
plt.title('True Clusters')  # Chart title
plt.xlabel('Feature 1')  # X-axis label
plt.ylabel('Feature 2')  # Y-axis label
plt.grid(True, alpha=0.3)  # Add grid

# Note: In real clustering, we don't know y_true!
# We're showing it here just to see what we're trying to discover

# Subplot 2: Hierarchical clustering discovered clusters
plt.subplot(1, 2, 2)  # 1 row, 2 columns, position 2 (right)

# Scatter plot colored by predicted cluster
plt.scatter(X[:, 0], X[:, 1], c=y_pred, cmap='viridis', s=50, alpha=0.7)
# c=y_pred: Color by cluster assignment
plt.title('Hierarchical Clustering (k=4, Ward)')  # Chart title
plt.xlabel('Feature 1')  # X-axis label
plt.ylabel('Feature 2')  # Y-axis label
plt.grid(True, alpha=0.3)  # Add grid

# Adjust layout
plt.tight_layout()
plt.show()  # Display both plots

# Interpretation:
# - Compare left plot (true clusters) with right plot (discovered clusters)
# - If they match well, hierarchical clustering found the right clusters
# - Ward linkage works well for spherical clusters


## Dendrogram Visualization

Let's create and analyze dendrograms for different linkage methods.


In [ ]:
# ============================================
# CREATING DENDOGRAMS: Visualizing Cluster Hierarchy
# ============================================

# Dendrograms show the hierarchical structure of clusters
# They help understand how clusters merge at different distances
# Different linkage methods produce different dendrograms

# Test different linkage methods
linkage_methods = ['ward', 'complete', 'average', 'single']
# - 'ward': Minimizes variance (good for spherical clusters)
# - 'complete': Maximum distance (compact clusters)
# - 'average': Average distance (balanced)
# - 'single': Minimum distance (can create long chains)

# Create figure with 2×2 grid of subplots
fig, axes = plt.subplots(2, 2, figsize=(16, 12))  # 2 rows, 2 columns

# Create dendrogram for each linkage method
for idx, method in enumerate(linkage_methods):
    # Calculate row and column position
    row = idx // 2  # Integer division (0 or 1)
    col = idx % 2  # Modulo (0 or 1)
    
    # ============================================
    # COMPUTING LINKAGE MATRIX
    # ============================================
    
    # linkage() computes the hierarchical clustering linkage matrix
    # This matrix encodes how clusters are merged at each step
    linkage_matrix = linkage(X, method=method)
    # X: Feature data
    # method: Linkage criterion ('ward', 'complete', 'average', 'single')
    # Returns: Linkage matrix (n-1 rows × 4 columns)
    #   Each row represents one merge: [cluster1, cluster2, distance, size]
    
    # ============================================
    # PLOTTING DENDOGRAM
    # ============================================
    
    # dendrogram() visualizes the hierarchical clustering as a tree
    # linkage_matrix: The computed linkage matrix
    # ax=axes[row, col]: Which subplot to use
    # truncate_mode='level': Truncate dendrogram to show only top levels
    # p=5: Show only 5 levels (prevents overcrowding)
    dendrogram(linkage_matrix, ax=axes[row, col], truncate_mode='level', p=5)
    
    # Label the subplot
    axes[row, col].set_title(f'Dendrogram ({method.capitalize()} Linkage)')  # Chart title
    axes[row, col].set_xlabel('Sample Index')  # X-axis: sample number
    axes[row, col].set_ylabel('Distance')  # Y-axis: distance at which clusters merge
    # Higher on y-axis = clusters merged later (more different)

# Adjust layout
plt.tight_layout()
plt.show()  # Display all dendrograms

# Interpretation:
# - Dendrogram shows cluster hierarchy (tree structure)
# - Height of merge = distance between clusters
# - Cut dendrogram at different heights to get different numbers of clusters
# - Different linkage methods produce different tree structures

# ============================================
# EXTRACTING CLUSTERS AT DIFFERENT LEVELS
# ============================================

# Hierarchical clustering allows extracting clusters at different "levels"
# We can use distance_threshold to cut the tree at different heights
print("\nClusters at different distance thresholds:")
for threshold in [2, 4, 6, 8]:
    # Create clustering with distance threshold (instead of n_clusters)
    clustering_thresh = AgglomerativeClustering(
        n_clusters=None,  # Don't specify number of clusters
        distance_threshold=threshold,  # Cut tree at this distance
        linkage='ward'  # Use ward linkage
    )
    # Clusters are formed when distance between clusters > threshold
    
    # Get cluster assignments
    labels_thresh = clustering_thresh.fit_predict(X)
    
    # Count number of clusters found
    n_clusters = len(np.unique(labels_thresh))  # Number of unique cluster labels
    print(f"  Distance threshold {threshold}: {n_clusters} clusters")
    # Lower threshold = more clusters (finer granularity)
    # Higher threshold = fewer clusters (coarser granularity)


## Comparing Linkage Methods

Let's compare different linkage criteria.


In [ ]:
# ============================================
# COMPARING LINKAGE METHODS: Which Works Best?
# ============================================

# Different linkage methods can produce very different clusterings
# We'll test all methods and compare their quality using silhouette score

# Test different linkage methods
linkage_methods = ['ward', 'complete', 'average', 'single']
results = []  # Store results for each method

# Test each linkage method
for method in linkage_methods:
    # Create clustering with this linkage method
    clustering = AgglomerativeClustering(n_clusters=4, linkage=method)
    
    # Get cluster assignments
    labels = clustering.fit_predict(X)  # Cluster labels for all samples
    
    # Calculate silhouette score (quality metric)
    sil_score = silhouette_score(X, labels)  # Average silhouette score
    # Range: -1 to 1 (higher = better clustering)
    
    # Store results
    results.append({
        'method': method,  # Linkage method name
        'silhouette': sil_score,  # Quality score
        'labels': labels  # Cluster assignments
    })
    
    print(f"{method.capitalize()} Linkage: Silhouette Score = {sil_score:.3f}")

# ============================================
# VISUALIZING COMPARISON: All Linkage Methods
# ============================================

# Create figure with 2×2 grid of subplots
fig, axes = plt.subplots(2, 2, figsize=(14, 12))  # 2 rows, 2 columns

# Plot clustering results for each method
for idx, result in enumerate(results):
    # Calculate row and column position
    row = idx // 2  # Integer division (0 or 1)
    col = idx % 2  # Modulo (0 or 1)
    
    # Scatter plot colored by cluster assignments
    axes[row, col].scatter(X[:, 0], X[:, 1], c=result['labels'], 
                          cmap='viridis', s=50, alpha=0.7)
    # X[:, 0]: First feature, X[:, 1]: Second feature
    # c=result['labels']: Color by cluster assignment
    # cmap='viridis': Color scheme
    
    # Set title with method name and silhouette score
    axes[row, col].set_title(f"{result['method'].capitalize()} (Silhouette: {result['silhouette']:.3f})")
    # Higher silhouette score = better clustering quality
    
    # Label axes
    axes[row, col].set_xlabel('Feature 1')  # X-axis label
    axes[row, col].set_ylabel('Feature 2')  # Y-axis label
    axes[row, col].grid(True, alpha=0.3)  # Add grid

# Adjust layout
plt.tight_layout()
plt.show()  # Display all plots

# Interpretation:
# - Compare silhouette scores to find best linkage method
# - Ward usually works best for spherical clusters
# - Single linkage can create long chains (chaining problem)
# - Complete linkage creates compact clusters
# - Average linkage is a balanced approach

# ============================================
# FINDING BEST LINKAGE METHOD
# ============================================

# Find linkage method with highest silhouette score
best_method = max(results, key=lambda x: x['silhouette'])
# max(..., key=...): Finds result with maximum silhouette score
# lambda x: x['silhouette']: Extract silhouette score for comparison

print(f"\nBest linkage method: {best_method['method']} (Silhouette: {best_method['silhouette']:.3f})")
# Display best method and its quality score


## Validation & Testing

Let's validate the clustering and evaluate performance.


In [ ]:
# Evaluate clustering
evaluation = evaluate_clustering(X, y_pred, algorithm='Hierarchical')
print("Clustering Evaluation:")
print(f"  Silhouette Score: {evaluation['silhouette_score']:.3f}")
print(f"  Number of clusters: {evaluation['n_clusters']}")

# Test different numbers of clusters
k_range = range(2, 8)
silhouette_scores = []

for k in k_range:
    clustering = AgglomerativeClustering(n_clusters=k, linkage='ward')
    labels = clustering.fit_predict(X)
    sil_score = silhouette_score(X, labels)
    silhouette_scores.append(sil_score)
    print(f"  k={k}: Silhouette Score = {sil_score:.3f}")

optimal_k = k_range[np.argmax(silhouette_scores)]
print(f"\nOptimal number of clusters: {optimal_k} (Silhouette: {max(silhouette_scores):.3f})")

# Plot silhouette scores
plt.figure(figsize=(8, 5))
plt.plot(k_range, silhouette_scores, 'bo-')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score vs Number of Clusters')
plt.grid(True, alpha=0.3)
plt.axvline(x=optimal_k, color='r', linestyle='--', label=f'Optimal k={optimal_k}')
plt.legend()
plt.tight_layout()
plt.show()

# Assertions
assert evaluation['silhouette_score'] > 0, "Silhouette score should be positive"
assert clustering.n_clusters == 4, "Expected 4 clusters"
print("\n✓ Validation checks passed")


## Real-World Application

Let's apply hierarchical clustering to the Iris dataset.


In [ ]:
# Load Iris dataset
iris = load_iris()
X_iris = iris.data
y_iris = iris.target

# Scale features
scaler = StandardScaler()
X_iris_scaled = scaler.fit_transform(X_iris)

# Apply Hierarchical Clustering
clustering_iris = AgglomerativeClustering(n_clusters=3, linkage='ward')
y_iris_pred = clustering_iris.fit_predict(X_iris_scaled)

# Evaluate
sil_score_iris = silhouette_score(X_iris_scaled, y_iris_pred)
print("Iris Dataset Clustering:")
print(f"  Number of clusters: {clustering_iris.n_clusters}")
print(f"  Silhouette Score: {sil_score_iris:.3f}")

# Create dendrogram
linkage_matrix_iris = linkage(X_iris_scaled, method='ward')
plt.figure(figsize=(12, 6))
dendrogram(linkage_matrix_iris, truncate_mode='level', p=3)
plt.title('Dendrogram for Iris Dataset (Ward Linkage)')
plt.xlabel('Sample Index')
plt.ylabel('Distance')
plt.tight_layout()
plt.show()

# Visualize clusters (using first two features)
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.scatter(X_iris[:, 0], X_iris[:, 1], c=y_iris, cmap='viridis', s=50, alpha=0.7)
plt.xlabel(iris.feature_names[0])
plt.ylabel(iris.feature_names[1])
plt.title('True Labels')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.scatter(X_iris[:, 0], X_iris[:, 1], c=y_iris_pred, cmap='viridis', s=50, alpha=0.7)
plt.xlabel(iris.feature_names[0])
plt.ylabel(iris.feature_names[1])
plt.title('Hierarchical Clustering (k=3, Ward)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Summary & Key Takeaways

### Key Concepts Learned

1. **Hierarchical Clustering Basics**
   - Builds tree of clusters (dendrogram)
   - Two approaches: agglomerative (bottom-up) and divisive (top-down)
   - No need to specify number of clusters in advance
   - Produces interpretable hierarchy

2. **Linkage Criteria**
   - **Single**: Minimum distance between clusters (chaining effect)
   - **Complete**: Maximum distance (compact clusters)
   - **Average**: Average distance (balanced)
   - **Ward**: Minimizes variance (spherical clusters, works with Euclidean)

3. **Dendrogram**
   - Visual representation of cluster hierarchy
   - Height represents distance at which clusters merge
   - Can extract clusters at any level by cutting dendrogram

4. **Best Practices**
   - Use Ward linkage with Euclidean distance (most common)
   - Scale features before clustering
   - Visualize dendrogram to choose number of clusters
   - Use silhouette score to validate clustering
   - Consider computational cost for large datasets

### When to Use Hierarchical Clustering

✅ **Good for:**
- Unknown number of clusters
- Need to explore cluster structure at multiple levels
- Want interpretable cluster hierarchy
- Small to medium datasets
- Non-spherical clusters
- When dendrogram visualization is helpful

❌ **Not ideal for:**
- Very large datasets (computationally expensive O(n³))
- When you know exact number of clusters (use K-Means)
- Real-time clustering (slow)
- Memory constraints (stores full distance matrix)
- When speed is critical

### Next Steps

- Compare with **K-Means** for known number of clusters
- Try **DBSCAN** for density-based clustering
- Use **PCA** before clustering for high-dimensional data
- Explore **Divisive Clustering** (top-down approach)
- Apply to **gene expression data** for biological clustering
